
# Hugging Face Transformers Library




## References

Hugging Face *Quicktour*: https://huggingface.co/docs/transformers/quicktour

Hugging Face *Run Inference with Pipelines tutorial*: https://huggingface.co/docs/transformers/pipeline_tutorial

Hugging Face *NLP Course, Chapter 2*: https://huggingface.co/learn/nlp-course/chapter2/1

## The Plan

The first thing we're going to do is get comfortable with the Hugging Face *Transformers* library for Python

Eventual goals:
* understand and explain how transformer models work
* create and adapt transformers for a specific purpose

For now:
* learn to *use* existing transformer models
* understand *what* they can do
* get a feel for pupular families of models and how they're related

We will dig into the internals later

## What is Hugging Face?

Hugging Face is a private company
* Founded in 2016 by French entrepreneurs Clément Delangue, Julien Chaumond, and Thomas Wolf
* Based in New York City

Provide a popular free, open-source Python library called **transformers** for NLP (and other) tasks

Host *hundreds of thousands of models* that we can use in your own programs

## Installing the transformers module

This is my favored way of installing packages from a Jupyter Notebook

If we have lots of Python distributions installed, it should use the right one

It may take a few minutes, but *we should only have to do this once*

In [1]:
import sys
!{sys.executable} -m pip install transformers accelerate

/bin/bash: /Users/sagar/School/Drake: No such file or directory


## Using the sentiment analysis pipeline

**Sentiment analysis** attempts to identify the overall feeling intended by the writer of some text

The creators of this model **trained** it on lots of examples of text that were labeled as either *positive* or *negative*

A **pipeline** is a series of steps for performing **inference**
* tokenize and preprocess the input text (more on this later)
* ask the model for a prediction
* post-process model's result and turn it into something we can use



We *are* specifying the kind of task: `sentiment-analysis`

We *are not* asking for a specific model, so it picks one of many it has by default

The first time we do this, it will have to download the model - this can take some time depending on your network connection

In [2]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

classifier("it is to build sentiment-aware applications with the transformers library!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.6030644774436951}]

**Test it out:** Try changing the input to get different labels/scores

## Hardware Acceleration

NLP Models are often *computationally intensive*

Algorithms can be sped up using special hardware like
* Graphics Processing Unit (GPU)
* Tensor Processing Unit (TPU)
* Other specialized devices


In [1]:
from accelerate import Accelerator
accelerator = Accelerator()
print( accelerator.device)

mps


Here's how we can tell the pipeline to use acceleration.

Examples from the Hugging Face documentation will often look like this.

In [4]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

classifier = pipeline("sentiment-analysis",device=device)

classifier("I love how easy it is to build sentiment-aware applications with the transformers library!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9984306693077087}]

## Working with batches of text

To get classifications of many different examples, pass in a list of strings.

In [ ]:
results = classifier(["It's really cool that we can get classifications for a whole batch of text",
                      "I wonder if the rest of the class will be this easy.",
                     "Spolier alert: it won't be."])
print(results)

[{'label': 'POSITIVE', 'score': 0.9991173148155212}, {'label': 'NEGATIVE', 'score': 0.9557347297668457}, {'label': 'NEGATIVE', 'score': 0.9962737560272217}]


Note that the results come back as a list of dictionaries, so we can manipulate it in the normal ways.

In [6]:
print("The sentence had",results[0]["label"],"sentiment, with a score of",results[0]["score"])

The sentence had POSITIVE sentiment, with a score of 0.9991173148155212


## Specifying a model



In [ ]:
classifier = pipeline("sentiment-analysis", model="SamLowe/roberta-base-go_emotions", device=device)
results = classifier(["It's really cool that we can get classifications for a whole batch of text",
                      "I wonder if the rest of the class will be this easy.",
                     "Spolier alert: it won't be.",
                      "If we know , we know"])
print(results)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'admiration', 'score': 0.5409093499183655}, {'label': 'surprise', 'score': 0.7312609553337097}, {'label': 'neutral', 'score': 0.7667409181594849}, {'label': 'neutral', 'score': 0.9300107955932617}]


This model has more than 2 labels unlike the previous one up above. Lemme check the labels for this model.

In [13]:
labels = classifier.model.config.id2label


print(labels)

{0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval', 5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment', 10: 'disapproval', 11: 'disgust', 12: 'embarrassment', 13: 'excitement', 14: 'fear', 15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness', 20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse', 25: 'sadness', 26: 'surprise', 27: 'neutral'}


In [14]:
labels = list(classifier.model.config.id2label.values())


print(labels)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


As we can see, the model is able to classify the text into different emotions.  27 in total.

In [ ]:


# Testing the models

# 1. Twitter-RoBERTa-base for Sentiment Analysis, "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 2. DistilRoBERTa Financial News Sentiment, "ProsusAI/finbert"

# 3. RoBERTa Spam Detection, "cardiffnlp/twitter-roberta-base-spam-detection"

# 4. BERT for News Topic Classification, "bert-base-uncased"





#  1. Twitter-RoBERTa-base for Sentiment Analysis


test_model1 = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
results = test_model1(["It's really cool that you can get classifications for a whole batch of text",
                      "I wonder if the rest of the class will be this easy.",
                     "Spolier alert: it won't be.",
                      "If you know , you know"])
print("Results for test_model1:", results)


test_model2 = pipeline("sentiment-analysis", model="ProsusAI/finbert")
results2 = test_model2(["It's really cool that we can get classifications for a whole batch of text",
                      "I wonder if the rest of the class will be this easy.",
                     "Spolier alert: it won't be.",
                      "If you know , you know"])
print("Results for test_model2:", results2)



# This Model turned out to be in private repository, I did not bother to request access.
"""
test_model3 = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-spam-detection")
results3 = test_model3(["It's really cool that you can get classifications for a whole batch of text",
                      "I wonder if the rest of the class will be this easy.",
                     "Spolier alert: it won't be.",
                      "If you know , you know"])
print("Results for test_model3:", results3) 


"""




test_model4 = pipeline("sentiment-analysis", model="bert-base-uncased")
results4 = test_model4(["It's really cool that you can get classifications for a whole batch of text",
                      "I wonder if the rest of the class will be this easy.",
                     "Spolier alert: it won't be.",
                      "If you know , you know"])
print("Results for test_model4:", results4)
    

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for test_model1: [{'label': 'positive', 'score': 0.9293923377990723}, {'label': 'neutral', 'score': 0.7285710573196411}, {'label': 'neutral', 'score': 0.647261917591095}, {'label': 'neutral', 'score': 0.6303098201751709}]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Results for test_model4: [{'label': 'LABEL_0', 'score': 0.5190641283988953}, {'label': 'LABEL_1', 'score': 0.5447249412536621}, {'label': 'LABEL_1', 'score': 0.5433131456375122}, {'label': 'LABEL_0', 'score': 0.5072804689407349}]


In [20]:
# printing out results only , above was too much information to concentrate on 

print("Results for test_model1:", results)
print("Results for test_model2:", results2)
print("Results for test_model4:", results4)


Results for test_model1: [{'label': 'positive', 'score': 0.9293923377990723}, {'label': 'neutral', 'score': 0.7285710573196411}, {'label': 'neutral', 'score': 0.647261917591095}, {'label': 'neutral', 'score': 0.6303098201751709}]
Results for test_model2: [{'label': 'neutral', 'score': 0.8397212624549866}, {'label': 'neutral', 'score': 0.9083960652351379}, {'label': 'neutral', 'score': 0.746833860874176}, {'label': 'neutral', 'score': 0.9279472827911377}]
Results for test_model4: [{'label': 'LABEL_0', 'score': 0.5190641283988953}, {'label': 'LABEL_1', 'score': 0.5447249412536621}, {'label': 'LABEL_1', 'score': 0.5433131456375122}, {'label': 'LABEL_0', 'score': 0.5072804689407349}]




---

## Part 1: Understanding roberta-base-go_emotions

### What is `roberta-base`?

**RoBERTa** stands for "Robustly Optimized BERT Pretraining Approach"   basically, it's an improved version of BERT that Facebook AI created.

Here's what makes it special:

**The Basics:**
- It's a **pre-trained language model** that understands English really well
- Created by **Facebook AI** (now Meta AI) and released in 2019
- It's **case-sensitive**, meaning it knows the difference between "English" and "english"
- Uses something called **masked language modeling (MLM)** - think of it like a fill-in-the-blank exercise where the model learns to predict missing words

**What it was trained on:**
The model learned from a MASSIVE amount of text - 160GB worth! This included:
1. **BookCorpus** - 11,038 unpublished books
2. **English Wikipedia** - all the articles (minus tables and lists)
3. **CC-News** - 63 million English news articles from 2016-2019
4. **OpenWebText** - a recreation of the dataset used to train GPT-2
5. **Stories** - a dataset with story-like text from CommonCrawl

**Why is it called "base"?**
"Base" means it's the standard-sized version (not the smaller or larger variants). It has 125 million parameters, which is a good balance between performance and speed.

**The key improvement over BERT:**
RoBERTa took BERT and made it better by training it longer, on more data, with bigger batches, and removing some training tricks that weren't actually helping. The result? Better performance on pretty much every task!

---

### What is `go_emotions`?

**GoEmotions** is a really cool dataset created by Google Research specifically for understanding emotions in text.

Here's what you need to know:

**The Dataset:**
- Contains **58,000 Reddit comments** that were manually labeled by real people
- All comments are in **English**
- Collected from popular English-language subreddits

**The Emotion Labels:**
Instead of just "positive" or "negative," this dataset recognizes **28 different categories**:
- **27 specific emotions** like admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, and surprise
- **1 neutral** category for when there's no strong emotion

**Why it's organized this way:**
- **12 positive emotions** (like joy, love, admiration)
- **11 negative emotions** (like anger, sadness, disgust)  
- **4 ambiguous emotions** (like surprise, which could be good or bad)
- **1 neutral** category

**Why it exists:**
Google created this to help build AI that can better understand human emotions in conversations. This is useful for:
- Making chatbots more empathetic
- Detecting harmful behavior online
- Understanding customer feedback
- Analyzing social media sentiment

**Important note:**
Some research has found that about 30% of the labels might not be perfect (emotions are subjective, after all!), but it's still one of the best datasets we have for fine-grained emotion detection.

**The `roberta-base-go_emotions` model:**
This is what you get when you take the roberta-base model and fine-tune it (train it more) specifically on the GoEmotions dataset. So it's really good at detecting those 28 different emotions in text!

---

## Part 2: Exploring Additional Models

I explored several different text classification models on HuggingFace. Here's what I found:

---

### Model 1: Twitter-RoBERTa-base for Sentiment Analysis
**Model:** `cardiffnlp/twitter-roberta-base-sentiment-latest`

**What is it for?**
This model is specifically designed to analyze sentiment (emotional tone) in tweets and social media posts. It classifies text into three categories:
- 0 = Negative
- 1 = Neutral  
- 2 = Positive

**Who made it?**
Created by **Cardiff NLP** (Natural Language Processing group at Cardiff University in Wales, UK). They're a research group that specializes in social media analysis.

**What data was it trained on?**
- First, it was pre-trained on **~124 million tweets** from January 2018 to December 2021
- Then it was fine-tuned using the **TweetEval benchmark** dataset
- This makes it really good at understanding informal language, hashtags, emojis, and the casual way people write on Twitter

**Is it based on another model?**
Yes! It's **fine-tuned from RoBERTa-base**. They took the standard RoBERTa model and specialized it for Twitter/social media text. This is a great example of transfer learning - starting with a model that understands general English, then teaching it to be an expert in social media language.

**Why it's cool:**
It's updated for modern language (2021 data), so it understands more recent slang and how people communicate online today. It's also been integrated into TweetNLP, a tool for analyzing tweets.

---

### Model 2: DistilRoBERTa Financial News Sentiment
**Model:** `mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis`

**What is it for?**
This model analyzes sentiment in **financial news and business articles**. It's designed to understand whether financial news is positive, negative, or neutral - super useful for investors and financial analysts!

**Who made it?**
Created by **mrm8488** (Manuel Romero), a prolific model creator on HuggingFace who has made hundreds of specialized models.

**What data was it trained on?**
- Fine-tuned on the **financial_phrasebank dataset**
- This dataset contains sentences from financial news articles that have been labeled by financial experts
- Achieves **98.23% accuracy** on the test set - that's really impressive!

**Is it based on another model?**
Yes! It's based on **distilroberta-base**, which is itself a smaller, faster version of RoBERTa. The "distil" part means it was "distilled" - compressed to be more efficient while keeping most of the performance. So the lineage is: BERT → RoBERTa → DistilRoBERTa → This financial sentiment model

**Why it's cool:**
Financial language is tricky because words can have different meanings in finance than in everyday use. For example, "aggressive growth" might sound negative normally, but it's often positive in finance. This model has learned those nuances!

---

### Model 3: RoBERTa Spam Detection
**Model:** `mshenoda/roberta-spam`

**What is it for?**
This model detects **spam messages** - whether a message is legitimate (ham) or spam. It works on:
- SMS text messages
- Telegram messages
- Emails
Output: 0 = ham (legitimate), 1 = spam

**Who made it?**
Created by **mshenoda** (Mohamed Shenoda), who made the code and training process publicly available on GitHub.

**What data was it trained on?**
This is interesting - they combined **three different spam datasets** to make it more robust:
1. **SMS Spam Collection** from Kaggle - classic SMS spam messages
2. **Telegram Spam Ham** - spam from the Telegram messaging app
3. **Enron Spam** - emails from the famous Enron email dataset

The data was split: 80% training, 10% validation, 10% testing. By combining multiple sources, the model learns to detect spam across different platforms!

**Is it based on another model?**
Yes! It's **fine-tuned from roberta-base**. They took the general RoBERTa model and specialized it for spam detection. This is another great example of taking a general-purpose model and teaching it a specific task.

**Why it's cool:**
Spam detection is actually really hard because spammers constantly change their tactics. By training on multiple types of spam (SMS, messaging apps, email), this model is more versatile and harder to fool.

---

### Model 4: BERT for News Topic Classification
**Model:** `fabriceyhc/bert-base-uncased-ag_news`

**What is it for?**
This model classifies **news articles into topics/categories**. It's trained on the AG News dataset, which has four main categories:
- World news
- Sports
- Business
- Science/Technology

**Who made it?**
Created by **fabriceyhc** (Fabrice Harel-Canada), though the model card doesn't have extensive documentation.

**What data was it trained on?**
- Trained on the **AG News dataset** (AG stands for "Academic News")
- This is a popular benchmark dataset with news articles from various sources
- Contains thousands of news headlines and descriptions

**Is it based on another model?**
Yes! It's **fine-tuned from bert-base-uncased**. 
- "BERT" = Bidirectional Encoder Representations from Transformers (the original transformer model from Google)
- "base" = standard size (not large or small)
- "uncased" = doesn't distinguish between uppercase and lowercase (treats "News" and "news" the same)

**Why it's cool:**
News classification is super useful for organizing news feeds, recommending articles, and analyzing media coverage. This model can instantly categorize any news article into one of the four major topics!

---

## Key Takeaways

**Pattern I noticed:**
Almost all these models follow the same pattern:
1. Start with a **base model** (BERT, RoBERTa, DistilRoBERTa)
2. **Fine-tune** it on a specific dataset for a specific task
3. The result is a **specialized expert** in that particular domain

**Why fine-tuning is awesome:**
Instead of training a model from scratch (which would take months and cost thousands of dollars), we can take a pre-trained model that already understands language and just teach it  specific task. It's like hiring someone who already speaks English fluently and just teaching them medical terminology, rather than teaching someone to speak English AND learn medicine from scratch!

**The model family tree:**
- **BERT** (Google, 2018) → Started it all
- **RoBERTa** (Facebook, 2019) → Improved BERT
- **DistilRoBERTa** → Smaller, faster RoBERTa
- **All the specialized models** → Fine-tuned versions for specific tasks

**Different domains need different models:**
- Social media language is different from formal writing → Twitter-RoBERTa
- Financial news has its own vocabulary → Financial sentiment model
- Spam changes across platforms → Multi-source spam detector
- News topics are distinct → News classifier

This exploration really shows how the HuggingFace ecosystem works: there's a foundation of powerful base models, and then thousands of people create specialized versions for every imaginable task

